# 05. Modeling and Evaluation

**Objective:** Compare exactly four models: **A — Naive baseline**, **B — Historical LightGBM**,
**C — Graph-enhanced LightGBM**, and **D — Graph Convolutional Network (GCN)**.
Models B and C retain the existing validation MAE hyperparameter search; the GCN
retains validation MAE early stopping. **Precision@20 and Capture@20 are the primary
hotspot decision metrics**; MAE and R² provide supplementary count-fit context.

Run from the `notebooks` directory after notebooks 03 and 04. The existing environment
needs GeoPandas, LightGBM, scikit-learn, joblib, PyTorch, and PyTorch Geometric.
No checkpoint or prediction from an earlier feature configuration is loaded for training.

## 1. Load data and define chronological splits

Load the verified modeling dataset, seed the existing models, and preserve the
2020–2023 training, 2024 validation, and exploratory 2025 evaluation periods.

In [1]:
import copy
import os
import platform
from pathlib import Path

# Avoid the local LightGBM/PyTorch native-thread crash when reloading models.
os.environ["OMP_NUM_THREADS"] = "1"

import geopandas as gpd
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import ParameterGrid
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

/Users/andrew/Documents/Projects/portfolio/3.Collisionprediction/.col/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/andrew/Documents/Projects/portfolio/3.Collisionprediction/.col/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:
model_data = pd.read_csv("../data/processed/model_data.csv")
model_data["quarter"] = pd.PeriodIndex(model_data["quarter"], freq="Q")
model_data = model_data.sort_values(["cell_id", "quarter"]).reset_index(drop=True)

assert model_data.shape == (8112, 13)
assert not model_data.duplicated(["cell_id", "quarter"]).any()
assert model_data["cell_id"].nunique() == 338

train = model_data[model_data["quarter"].between("2020Q1", "2023Q4")].copy()
val = model_data[model_data["quarter"].between("2024Q1", "2024Q4")].copy()
test = model_data[model_data["quarter"].between("2025Q1", "2025Q4")].copy()

for frame, start, end in [
    (train, "2020Q1", "2023Q4"),
    (val, "2024Q1", "2024Q4"),
    (test, "2025Q1", "2025Q4"),
]:
    expected = pd.period_range(start, end, freq="Q")
    assert sorted(frame["quarter"].unique()) == list(expected)
    assert frame.groupby("quarter").size().eq(338).all()
assert len(train) + len(val) + len(test) == len(model_data)
assert train["quarter"].max() < val["quarter"].min() < test["quarter"].min()
print("Train / validation / test rows:", len(train), len(val), len(test))

Train / validation / test rows: 5408 1352 1352


## 2. Model A — Naive historical baseline

Use the previous quarter's collision count (`lag_1`) directly, without fitting a model.
All models forecast one quarter ahead using history available before each quarter.

In [3]:
val["baseline_pred"] = val["lag_1"]

## 3. Models B and C — LightGBM

Model B uses four historical features. Model C adds four road-network features.
Both use the original hyperparameter grid, fitted on 2020–2023 and selected on
2024 validation MAE. The comparison below reports one selected model per feature set.

In [4]:
historical_cols = [
    "lag_1",
    "lag_4",
    "rolling_mean_4",
    "quarter_of_year"
]
graph_cols = [
    "road_length_m",
    "major_road_share",
    "intersection_density",
    "mean_streets_per_node"
]
model_c_features = historical_cols + graph_cols

assert len(historical_cols) == 4
assert len(model_c_features) == 8
assert np.isfinite(model_data[model_c_features].to_numpy()).all()

In [5]:
param_grid = {
    "n_estimators": [100, 200],
    "num_leaves": [7, 15, 31],
    "min_child_samples": [20, 50]
}

def tune_model(features, train_data, val_data):
    best_mae = float("inf")
    best_model = None
    best_params = None

    for params in ParameterGrid(param_grid):
        model = LGBMRegressor(
            **params,
            learning_rate=0.05,
            random_state=SEED,
            verbosity=-1,
            n_jobs=1
        )

        model.fit(train_data[features], train_data["collisions"])

        predictions = np.maximum(
            model.predict(val_data[features]), 0
        )

        mae = mean_absolute_error(
            val_data["collisions"], predictions
        )

        if mae < best_mae:
            best_mae = mae
            best_model = model
            best_params = params

    return best_model, best_mae, best_params


# Tune both using the same search space
best_model_b, best_mae_b, params_b = tune_model(historical_cols, train, val)
best_model_c, best_mae_c, params_c = tune_model(
    model_c_features, train, val
)

print("Model B:", best_mae_b, params_b)
print("Model C:", best_mae_c, params_c)

Model B: 2.3345403907682414 {'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 7}
Model C: 2.1496895383026557 {'min_child_samples': 50, 'n_estimators': 200, 'num_leaves': 31}


In [6]:
val["model_b_pred"] = np.maximum(best_model_b.predict(val[historical_cols]), 0)
val["model_c_pred"] = np.maximum(best_model_c.predict(val[model_c_features]), 0)

## 4. Hotspot evaluation

For each quarter, rank cells by predicted collisions and compare the top 20 with
the observed top 20. Ties use `cell_id` for reproducibility. **Precision@20** is the
fraction of selected cells in the observed top 20; **Capture@20** is the fraction
of all collisions occurring in selected cells. Average both metrics across quarters.

In [7]:
def evaluate_model(df, prediction_col, k=20):
    assert k == 20, "This experiment reports Precision@20 and Capture@20."
    assert not df.empty
    assert not df.duplicated(["cell_id", "quarter"]).any()
    assert np.isfinite(df[["collisions", prediction_col]].to_numpy()).all()
    results = []

    for quarter, group in df.groupby("quarter", sort=True):
        assert len(group) >= k
        predicted_top = group.sort_values(
            [prediction_col, "cell_id"], ascending=[False, True]
        ).head(k)
        actual_top = group.sort_values(
            ["collisions", "cell_id"], ascending=[False, True]
        ).head(k)
        overlap = len(set(predicted_top["cell_id"]) & set(actual_top["cell_id"]))
        total = group["collisions"].sum()
        results.append({
            "quarter": str(quarter),
            "precision_at_20": overlap / k,
            "capture_at_20": predicted_top["collisions"].sum() / total if total else 0.0
        })

    quarterly_results = pd.DataFrame(results)
    return {
        "Precision@20": quarterly_results["precision_at_20"].mean(),
        "Capture@20": quarterly_results["capture_at_20"].mean(),
        "MAE": mean_absolute_error(df["collisions"], df[prediction_col]),
        "R²": r2_score(df["collisions"], df[prediction_col])
    }

## 5. Model D — Graph Convolutional Network

The same eight features as Model C describe 338 grid-cell nodes. Edges connect
cells sharing a boundary segment; corner-only contacts are excluded. The existing
two-layer GCN (32 and 16 hidden units), Adam optimizer, MSE training loss, validation
MAE checkpoint selection, and early stopping are retained. Scaling uses training
rows only. Every run initializes a new eight-input model and retrains it.

In [8]:


print("Architecture:", platform.machine())
print("PyTorch:", torch.__version__)
print("PyTorch Geometric:", torch_geometric.__version__)
print("MPS available:", torch.backends.mps.is_available())

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print("Selected device:", device)

Architecture: arm64
PyTorch: 2.15.0
PyTorch Geometric: 2.8.1
MPS available: True
Selected device: mps


In [9]:
grid = gpd.read_file("../data/processed/manhattan_grid.gpkg").to_crs("EPSG:32618")
grid = grid[["cell_id", "geometry"]].sort_values("cell_id").reset_index(drop=True)
assert len(grid) == 338 and grid["cell_id"].is_unique
cell_ids = grid["cell_id"].tolist()
id_to_idx = {cell_id: i for i, cell_id in enumerate(cell_ids)}
assert set(cell_ids) == set(model_data["cell_id"])

neighbors = gpd.sjoin(grid, grid, how="inner", predicate="touches")
geometry = grid.set_index("cell_id").geometry
# Keep shared-edge neighbors, excluding corner-only contacts.
edge_pairs = sorted({
    (a, b)
    for a, b in zip(neighbors["cell_id_left"], neighbors["cell_id_right"])
    if a < b
    and geometry.loc[a].boundary.intersection(geometry.loc[b].boundary).length > 1
})
# PyTorch Geometric needs both directions, mapped from cell IDs to row indices.
directed_edges = edge_pairs + [(b, a) for a, b in edge_pairs]
edge_index = torch.tensor([
    [id_to_idx[a] for a, b in directed_edges],
    [id_to_idx[b] for a, b in directed_edges]
], dtype=torch.long)
assert edge_index.shape == (2, 2 * len(edge_pairs))
assert edge_index.numel() > 0
assert edge_index.min().item() >= 0 and edge_index.max().item() < len(cell_ids)
assert set(map(tuple, edge_index.T.tolist())) == {
    (id_to_idx[a], id_to_idx[b]) for a, b in directed_edges
}
assert set(directed_edges) == {(b, a) for a, b in directed_edges}
connected_cells = {cell_id for pair in edge_pairs for cell_id in pair}
print("Nodes:", len(cell_ids), "| Shared edges:", len(edge_pairs))
print("Isolated cells:", len(set(cell_ids) - connected_cells))
print("Edge index shape:", edge_index.shape)

Nodes: 338 | Shared edges: 592
Isolated cells: 0
Edge index shape: torch.Size([2, 1184])


In [10]:
gnn_features = historical_cols + graph_cols

assert len(gnn_features) == 8

# Fit scaling ONLY on training data
scaler = StandardScaler()
scaler.fit(train[gnn_features].to_numpy(dtype=np.float32))

def make_graph_batch(df, scaler):
    graphs = []

    for quarter in sorted(df["quarter"].unique()):
        q_rows = df[df["quarter"] == quarter]
        assert q_rows["cell_id"].is_unique
        assert set(q_rows["cell_id"]) == set(cell_ids)
        q_data = (
            df[df["quarter"] == quarter]
            .set_index("cell_id")
            .reindex(cell_ids)
        )

        assert q_data[gnn_features + ["collisions"]].notna().all().all()

        x = scaler.transform(
            q_data[gnn_features].to_numpy(dtype=np.float32)
        )

        y = q_data["collisions"].to_numpy(dtype=np.float32)

        graphs.append(Data(
            x=torch.tensor(x, dtype=torch.float32),
            edge_index=edge_index,
            y=torch.tensor(y, dtype=torch.float32)
        ))

    return Batch.from_data_list(graphs)


train_batch = make_graph_batch(train, scaler)
val_batch = make_graph_batch(val, scaler)


print("Training graphs:", train_batch.num_graphs)
print("Validation graphs:", val_batch.num_graphs)
print("Training feature shape:", train_batch.x.shape)

Training graphs: 16
Validation graphs: 4
Training feature shape: torch.Size([5408, 8])


In [11]:
class RoadRiskGCN(nn.Module):
    def __init__(self, num_features):
        super().__init__()

        self.conv1 = GCNConv(num_features, 32)
        self.conv2 = GCNConv(32, 16)
        self.output = nn.Linear(16, 1)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        # Collision counts cannot be negative
        return F.softplus(self.output(x)).squeeze(-1)


torch.manual_seed(SEED)

gnn = RoadRiskGCN(len(gnn_features)).to(device)

train_batch = train_batch.to(device)
val_batch = val_batch.to(device)

optimizer = torch.optim.Adam(
    gnn.parameters(),
    lr=0.01,
    weight_decay=1e-4
)

best_mae = float("inf")
best_state = None
best_epoch = 0
patience = 35

for epoch in range(1, 2001):
    gnn.train()
    optimizer.zero_grad()

    predictions = gnn(
        train_batch.x,
        train_batch.edge_index
    )

    loss = F.mse_loss(predictions, train_batch.y)
    loss.backward()
    optimizer.step()

    # Evaluate on 2024 validation data
    gnn.eval()

    with torch.no_grad():
        val_predictions = gnn(
            val_batch.x,
            val_batch.edge_index
        )

        val_mae = F.l1_loss(
            val_predictions,
            val_batch.y
        ).item()

    if val_mae < best_mae - 1e-4:
        best_mae = val_mae
        best_epoch = epoch
        best_state = copy.deepcopy(gnn.state_dict())

    if epoch % 25 == 0:
        print(
            f"Epoch {epoch} | "
            f"Train MSE: {loss.item():.3f} | "
            f"Validation MAE: {val_mae:.3f}"
        )

    if epoch - best_epoch >= patience:
        print("Early stopping")
        break

assert best_state is not None, "No finite validation checkpoint was selected."
gnn.load_state_dict(best_state)

print(f"Best epoch: {best_epoch}")
print(f"Best validation MAE: {best_mae:.3f}")

Epoch 25 | Train MSE: 53.818 | Validation MAE: 3.695
Epoch 50 | Train MSE: 34.580 | Validation MAE: 3.385
Epoch 75 | Train MSE: 30.920 | Validation MAE: 3.293
Epoch 100 | Train MSE: 29.277 | Validation MAE: 3.226
Epoch 125 | Train MSE: 28.092 | Validation MAE: 3.163
Epoch 150 | Train MSE: 27.173 | Validation MAE: 3.138
Epoch 175 | Train MSE: 26.490 | Validation MAE: 3.116
Epoch 200 | Train MSE: 25.943 | Validation MAE: 3.096
Epoch 225 | Train MSE: 25.376 | Validation MAE: 3.067
Epoch 250 | Train MSE: 24.569 | Validation MAE: 3.026
Epoch 275 | Train MSE: 23.857 | Validation MAE: 2.985
Epoch 300 | Train MSE: 23.287 | Validation MAE: 2.952
Epoch 325 | Train MSE: 22.752 | Validation MAE: 2.937
Epoch 350 | Train MSE: 22.136 | Validation MAE: 2.923
Epoch 375 | Train MSE: 21.533 | Validation MAE: 2.903
Epoch 400 | Train MSE: 21.025 | Validation MAE: 2.865
Epoch 425 | Train MSE: 20.525 | Validation MAE: 2.846
Epoch 450 | Train MSE: 20.198 | Validation MAE: 2.838
Epoch 475 | Train MSE: 19.960 |

In [12]:
def predict_gnn(df, model, scaler):
    batch = make_graph_batch(df, scaler).to(next(model.parameters()).device)
    model.eval()
    with torch.no_grad():
        predictions = model(batch.x, batch.edge_index).cpu().numpy()
    # Batch rows are ordered by quarter, then the saved grid-cell node order.
    pred_index = pd.MultiIndex.from_product(
        [sorted(df["quarter"].unique()), cell_ids], names=["quarter", "cell_id"]
    )
    pred_lookup = pd.Series(predictions, index=pred_index)
    aligned = pred_lookup.reindex(pd.MultiIndex.from_frame(df[["quarter", "cell_id"]]))
    assert aligned.notna().all()
    return aligned.to_numpy()

val["model_d_pred"] = predict_gnn(val, gnn, scaler)
model_predictions = {
    "Model A — Naive": "baseline_pred",
    "Model B — Historical LightGBM": "model_b_pred",
    "Model C — Graph-enhanced LightGBM": "model_c_pred",
    "Model D — GCN": "model_d_pred"
}
comparison = pd.DataFrame({
    name: evaluate_model(val, col) for name, col in model_predictions.items()
}).T
print("2024 validation — primary metrics: Precision@20 and Capture@20")
display(comparison.round(3))

2024 validation — primary metrics: Precision@20 and Capture@20


,Precision@20,Capture@20,MAE,R²
Model A — Naive,0.550,0.189,2.757,0.766
Model B — Historical LightGBM,0.550,0.186,2.335,0.821
Model C — Graph-enhanced LightGBM,0.612,0.191,2.150,0.851
Model D — GCN,0.525,0.180,2.729,0.786


## 6. Retrospective evaluation on 2025

Keep the selected LightGBM settings. Refit all learned models on 2020–2024 before
2025 evaluation. The final GCN starts fresh with a new scaler fitted on those same
rows and trains for the epoch selected using 2024 validation. No 2025 outcomes are
used for fitting or epoch selection.

Each forecast uses actual history available before that quarter. These are
retrospective out-of-time evaluations because the periods were already inspected.
The quarterly table includes oracle Capture@20: coverage obtained by selecting the
actual top 20 cells. This retrospective upper bound is not a forecasting strategy.

In [13]:
train_final = pd.concat([train, val])

final_model_b = LGBMRegressor(
    **params_b,
    learning_rate=0.05,
    random_state=SEED,
    verbosity=-1,
    n_jobs=1
)

final_model_c = LGBMRegressor(
    **params_c,
    learning_rate=0.05,
    random_state=SEED,
    verbosity=-1,
    n_jobs=1
)

final_model_b.fit(
    train_final[historical_cols],
    train_final["collisions"]
)

final_model_c.fit(
    train_final[model_c_features],
    train_final["collisions"]
)

final_scaler = StandardScaler()
final_scaler.fit(train_final[gnn_features].to_numpy(dtype=np.float32))
final_train_batch = make_graph_batch(train_final, final_scaler).to(device)

torch.manual_seed(SEED)
final_gnn = RoadRiskGCN(len(gnn_features)).to(device)
final_optimizer = torch.optim.Adam(final_gnn.parameters(), lr=0.01, weight_decay=1e-4)

for epoch in range(1, best_epoch + 1):
    final_gnn.train()
    final_optimizer.zero_grad()
    predictions = final_gnn(final_train_batch.x, final_train_batch.edge_index)
    loss = F.mse_loss(predictions, final_train_batch.y)
    loss.backward()
    final_optimizer.step()

final_gnn.eval()

RoadRiskGCN(
  (conv1): GCNConv(8, 32)
  (conv2): GCNConv(32, 16)
  (output): Linear(in_features=16, out_features=1, bias=True)
)

In [14]:
test["baseline_pred"] = test["lag_1"]
test["model_b_pred"] = np.maximum(final_model_b.predict(test[historical_cols]), 0)
test["model_c_pred"] = np.maximum(final_model_c.predict(test[model_c_features]), 0)
test["model_d_pred"] = predict_gnn(test, final_gnn, final_scaler)

quarterly_rows = []
for quarter, rows in test.groupby("quarter", sort=True):
    actual_top = rows.sort_values(["collisions", "cell_id"], ascending=[False, True]).head(20)
    total = rows["collisions"].sum()
    oracle_capture = actual_top["collisions"].sum() / total if total else 0.0
    for name, column in model_predictions.items():
        quarterly_rows.append({
            "Quarter": str(quarter), "Model": name,
            **evaluate_model(rows, column), "Oracle Capture@20": oracle_capture
        })
quarterly_results = pd.DataFrame(quarterly_rows)

# Reuse the quarterly hotspot scores; MAE and R² retain the existing pooled definition.
test_results_df = quarterly_results.groupby("Model")[["Precision@20", "Capture@20"]].mean()
test_results_df = test_results_df.reindex(list(model_predictions))
for name, column in model_predictions.items():
    test_results_df.loc[name, "MAE"] = mean_absolute_error(test["collisions"], test[column])
    test_results_df.loc[name, "R²"] = r2_score(test["collisions"], test[column])

display(test_results_df.round(3))
display(quarterly_results.set_index(["Quarter", "Model"]).round(3))

,Precision@20,Capture@20,MAE,R²
Model,,,,
Model A — Naive,0.462,0.158,3.363,0.722
Model B — Historical LightGBM,0.512,0.163,2.799,0.797
Model C — Graph-enhanced LightGBM,0.562,0.168,2.801,0.796
Model D — GCN,0.500,0.157,3.289,0.737


Precision@20  Capture@20    MAE  \
Quarter Model                                                                
2025Q1  Model A — Naive                            0.45       0.167  3.092   
        Model B — Historical LightGBM              0.55       0.175  2.014   
        Model C — Graph-enhanced LightGBM          0.60       0.180  2.071   
        Model D — GCN                              0.50       0.166  2.490   
2025Q2  Model A — Naive                            0.30       0.141  4.192   
        Model B — Historical LightGBM              0.25       0.135  3.773   
        Model C — Graph-enhanced LightGBM          0.40       0.147  3.657   
        Model D — GCN                              0.35       0.136  3.796   
2025Q3  Model A — Naive                            0.55       0.166  3.154   
        Model B — Historical LightGBM              0.60       0.176  2.833   
        Model C — Graph-enhanced LightGBM          0.60       0.173  2.872   
        Model D — GCN                              0.50       0.159  3.541   
2025Q4  Model A — Naive                            0.55       0.158  3.015   
        Model B — Historical LightGBM              0.65       0.164  2.576   
        Model C — Graph-enhanced LightGBM          0.65       0.171  2.603   
        Model D — GCN                              0.65       0.169  3.328   

                                              R²  Oracle Capture@20  
Quarter Model                                                        
2025Q1  Model A — Naive                    0.635              0.190  
        Model B — Historical LightGBM      0.854              0.190  
        Model C — Graph-enhanced LightGBM  0.837              0.190  
        Model D — GCN                      0.779              0.190  
2025Q2  Model A — Naive                    0.625              0.187  
        Model B — Historical LightGBM      0.672              0.187  
        Model C — Graph-enhanced LightGBM  0.691              0.187  
        Model D — GCN                      0.663              0.187  
2025Q3  Model A — Naive                    0.779              0.195  
        Model B — Historical LightGBM      0.816              0.195  
        Model C — Graph-enhanced LightGBM  0.809              0.195  
        Model D — GCN                      0.730              0.195  
2025Q4  Model A — Naive                    0.795              0.183  
        Model B — Historical LightGBM      0.851              0.183  
        Model C — Graph-enhanced LightGBM  0.847              0.183  
        Model D — GCN                      0.775              0.183

In [15]:
# Keep the existing prediction filename and columns used by app.py; include Model D.
test[["cell_id", "quarter", "collisions", *model_predictions.values()]].to_csv(
    "../data/processed/test_predictions.csv", index=False
)

## 7. Save the final fitted models

Save the refitted GCN and its matching 2020–2024 scaler. Keep the selected epoch
and original validation MAE as metadata. Reload each final model once to compare
its predictions with the final 2025 predictions.

In [16]:
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)


model_config = {
    "historical_cols": historical_cols,
    "graph_cols": graph_cols,
    "gnn_features": gnn_features,
    "target": "collisions",
    "model_b_params": params_b,
    "model_c_params": params_c,
    "seed": SEED,
    "lightgbm_fit_period": ["2020Q1", "2024Q4"],
    "gnn_fit_period": ["2020Q1", "2024Q4"],
    "validation_period": ["2024Q1", "2024Q4"],
    "evaluation_period": ["2025Q1", "2025Q4"],
    "evaluation_status": "exploratory"
}
# Lists and tensors keep this checkpoint portable with weights_only=True.
gnn_checkpoint = {
    "model_state_dict": {name: value.detach().cpu() for name, value in final_gnn.state_dict().items()},
    "architecture": {"num_features": 8, "hidden_channels": [32, 16]},
    "gnn_features": gnn_features,
    "cell_ids": cell_ids,
    "edge_index": edge_index.cpu(),
    "scaler": {
        "mean_": final_scaler.mean_.tolist(),
        "scale_": final_scaler.scale_.tolist(),
        "var_": final_scaler.var_.tolist(),
        "n_features_in_": int(final_scaler.n_features_in_),
        "n_samples_seen_": int(final_scaler.n_samples_seen_)
    },
    "best_epoch": best_epoch,
    "validation_mae": best_mae,
    "seed": SEED,
    "fit_period": ["2020Q1", "2024Q4"],
    "selection_fit_period": ["2020Q1", "2023Q4"],
    "training_device": str(device),
    "validation_period": ["2024Q1", "2024Q4"],
    "evaluation_status": "2025 exploratory"
}
joblib.dump(final_model_b, MODEL_DIR / "model_b.pkl")
joblib.dump(final_model_c, MODEL_DIR / "model_c.pkl")
joblib.dump(model_config, MODEL_DIR / "model_config.pkl")
torch.save(gnn_checkpoint, MODEL_DIR / "gnn_model.pth")
print("Saved the current 4-feature Model B, 8-feature Model C, and 8-feature GCN with scaler.")

Saved the current 4-feature Model B, 8-feature Model C, and 8-feature GCN with scaler.


In [17]:
# One prediction comparison per saved model.
from threadpoolctl import threadpool_limits

with threadpool_limits(limits=1):
    for filename, features, prediction_col in [
        ("model_b.pkl", historical_cols, "model_b_pred"),
        ("model_c.pkl", model_c_features, "model_c_pred")
    ]:
        loaded = joblib.load(MODEL_DIR / filename)
        assert np.allclose(np.maximum(loaded.predict(test[features]), 0), test[prediction_col])

    checkpoint = torch.load(MODEL_DIR / "gnn_model.pth", map_location="cpu", weights_only=True)
    assert checkpoint["cell_ids"] == cell_ids
    restored_scaler = StandardScaler()
    for name in ["mean_", "scale_", "var_"]:
        setattr(restored_scaler, name, np.asarray(checkpoint["scaler"][name]))
    restored_scaler.n_features_in_ = checkpoint["scaler"]["n_features_in_"]
    restored_scaler.n_samples_seen_ = checkpoint["scaler"]["n_samples_seen_"]

    loaded_gnn = RoadRiskGCN(len(gnn_features))
    loaded_gnn.load_state_dict(checkpoint["model_state_dict"])
    predictions = predict_gnn(test, loaded_gnn, restored_scaler)
    assert np.allclose(predictions, test["model_d_pred"], rtol=1e-4, atol=1e-4)

print("Saved models reproduce the final 2025 predictions.")

Saved models reproduce the final 2025 predictions.


## 8. Retrospective rolling-origin backtests

| Training | Validation | Evaluation |
| --- | --- | --- |
| 2020–2021 | 2022 | 2023 |
| 2020–2022 | 2023 | 2024 |
| 2020–2023 | 2024 | 2025 |

Reuse the existing grid search, then refit on training plus validation. The last
fold reuses the main 2025 results. Backtest models do not overwrite saved models.

In [18]:
folds = [("2021Q4", 2022, 2023), ("2022Q4", 2023, 2024), ("2023Q4", 2024, 2025)]
backtest_rows = []
for train_end, validation_year, evaluation_year in folds:
    if evaluation_year == 2025:
        fold_results = test_results_df.loc[list(model_predictions)[:3]]
    else:
        fold_train = model_data[model_data["quarter"].between("2020Q1", train_end)].copy()
        fold_val = model_data[model_data["quarter"].dt.year == validation_year].copy()
        fold_test = model_data[model_data["quarter"].dt.year == evaluation_year].copy()
        _, _, fold_params_b = tune_model(historical_cols, fold_train, fold_val)
        _, _, fold_params_c = tune_model(model_c_features, fold_train, fold_val)
        fold_refit = pd.concat([fold_train, fold_val])

        fold_model_b = LGBMRegressor(**fold_params_b, learning_rate=0.05, random_state=SEED, verbosity=-1, n_jobs=1)
        fold_model_c = LGBMRegressor(**fold_params_c, learning_rate=0.05, random_state=SEED, verbosity=-1, n_jobs=1)
        fold_model_b.fit(fold_refit[historical_cols], fold_refit["collisions"])
        fold_model_c.fit(fold_refit[model_c_features], fold_refit["collisions"])
        fold_test["baseline_pred"] = fold_test["lag_1"]
        fold_test["model_b_pred"] = np.maximum(fold_model_b.predict(fold_test[historical_cols]), 0)
        fold_test["model_c_pred"] = np.maximum(fold_model_c.predict(fold_test[model_c_features]), 0)
        fold_results = pd.DataFrame({
            name: evaluate_model(fold_test, column)
            for name, column in list(model_predictions.items())[:3]
        }).T

    for name, scores in fold_results.iterrows():
        backtest_rows.append({
            "Training": "2020Q1–" + train_end, "Validation": validation_year,
            "Evaluation": evaluation_year, "Model": name, **scores.to_dict()
        })
backtest_results = pd.DataFrame(backtest_rows)
display(backtest_results.set_index(["Evaluation", "Model"]).round(3))

Training  Validation  \
Evaluation Model                                                          
2023       Model A — Naive                    2020Q1–2021Q4        2022   
           Model B — Historical LightGBM      2020Q1–2021Q4        2022   
           Model C — Graph-enhanced LightGBM  2020Q1–2021Q4        2022   
2024       Model A — Naive                    2020Q1–2022Q4        2023   
           Model B — Historical LightGBM      2020Q1–2022Q4        2023   
           Model C — Graph-enhanced LightGBM  2020Q1–2022Q4        2023   
2025       Model A — Naive                    2020Q1–2023Q4        2024   
           Model B — Historical LightGBM      2020Q1–2023Q4        2024   
           Model C — Graph-enhanced LightGBM  2020Q1–2023Q4        2024   

                                              Precision@20  Capture@20    MAE  \
Evaluation Model                                                                
2023       Model A — Naive                           0.600       0.198  2.632   
           Model B — Historical LightGBM             0.538       0.187  2.398   
           Model C — Graph-enhanced LightGBM         0.650       0.201  2.311   
2024       Model A — Naive                           0.550       0.189  2.757   
           Model B — Historical LightGBM             0.550       0.184  2.351   
           Model C — Graph-enhanced LightGBM         0.650       0.194  2.181   
2025       Model A — Naive                           0.462       0.158  3.363   
           Model B — Historical LightGBM             0.512       0.163  2.799   
           Model C — Graph-enhanced LightGBM         0.562       0.168  2.801   

                                                 R²  
Evaluation Model                                     
2023       Model A — Naive                    0.787  
           Model B — Historical LightGBM      0.796  
           Model C — Graph-enhanced LightGBM  0.827  
2024       Model A — Naive                    0.766  
           Model B — Historical LightGBM      0.819  
           Model C — Graph-enhanced LightGBM  0.857  
2025       Model A — Naive                    0.722  
           Model B — Historical LightGBM      0.797  
           Model C — Graph-enhanced LightGBM  0.796

## 9. Summary

The original LightGBM grid and settings are unchanged. All learned models are
refitted through 2024 for the final retrospective comparison. The quarterly table
includes oracle coverage, and backtests reuse the existing tuning and evaluation
functions. The corrected GCN refit and new tables need execution; their results
have not been generated during this edit.